# Hackathon ONE — Alura + Oracle | Equipo G9-LATAM-69
## Análisis inteligente de consumo energético residencial

**Workstream: Ciencia de Datos.** Notebook autocontenido que cubre todos los entregables del reto:

1. EDA (exploración y limpieza)
2. Análisis de patrones de consumo
3. Transformación de variables
4. Modelos supervisados
5. Evaluación con métricas
6. Recomendaciones
7. Serialización (ONNX)

Clasifica cada vivienda en **Eficiente / Moderado / Ineficiente**, devuelve la **probabilidad**,
**recomendaciones** y el **costo mensual** con tarifa **\$0,75/kWh**. Salida en el contrato JSON del reto.

> **Criterio central — evitar *data leakage*.** La eficiencia NO se define con la columna
> `eficiencia_energetica` (se demuestra abajo que no refleja consumo real) sino como el **residual**
> entre consumo real y esperado, separando *normalizadores* (estructura) de *palancas* (hábitos).

## 1. Configuración e imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
RANDOM_STATE = 42
TARIFA_REFERENCIA = 0.75  # $/kWh, valor sugerido por el reto
pd.set_option("display.width", 120); np.set_printoptions(suppress=True)
print("Entorno listo.")

## 2. Carga y limpieza de datos (EDA inicial)

Dataset `consumo_energia_sudamerica.csv` (30.000 viviendas, 5 países), separado por `;`,
codificado en Latin-1. Renombramos columnas a ASCII para evitar problemas de codificación.

In [ ]:
COLS = ["id_vivienda","pais","ciudad","tipo_vivienda","personas","superficie_m2","banos",
"equipos_electricos","consumo_mes_kwh","consumo_hora_punta_kwh","consumo_hora_valle_kwh",
"consumo_hora_normal_kwh","horario_mayor_consumo","refrigeradores","televisores","computadores",
"lavadora","secadora","aires_acondicionados","estufas_electricas","temperatura_promedio",
"medidor_inteligente","vehiculo_electrico","porcentaje_led","panel_solar","tarifa_clp_kwh_x100",
"valor_kwh_clp","gasto_mensual_clp","perfil_uso","estacion","eficiencia_energetica","recomendacion"]
df = pd.read_csv("consumo_energia_sudamerica.csv", sep=";", encoding="latin-1")
df.columns = COLS
print("Dimensiones:", df.shape)
print("Nulos totales:", df.isnull().sum().sum(), "| Duplicados:", df.duplicated().sum())
df.head()

In [ ]:
df[["personas","superficie_m2","equipos_electricos","consumo_mes_kwh",
    "consumo_hora_punta_kwh","porcentaje_led","temperatura_promedio"]].describe().round(1)

## 3. Análisis de patrones de consumo

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
df["consumo_mes_kwh"].hist(bins=40, ax=ax[0], color="#2E5A87")
ax[0].set_title("Distribución del consumo mensual (kWh)"); ax[0].set_xlabel("kWh/mes")
df.groupby("personas")["consumo_mes_kwh"].mean().plot(kind="bar", ax=ax[1], color="#3E7CB1")
ax[1].set_title("Consumo medio por nº de personas"); ax[1].set_ylabel("kWh/mes")
df.groupby("tipo_vivienda")["consumo_mes_kwh"].mean().plot(kind="bar", ax=ax[2], color="#5B9BD5")
ax[2].set_title("Consumo medio por tipo de vivienda"); ax[2].set_ylabel("kWh/mes")
plt.tight_layout(); plt.show()

In [ ]:
print("Consumo medio por país (kWh/mes):")
print(df.groupby("pais")["consumo_mes_kwh"].mean().round(1).sort_values(ascending=False))
corr = df[["personas","superficie_m2","equipos_electricos","banos","refrigeradores",
           "aires_acondicionados","temperatura_promedio","consumo_mes_kwh"]].corr()["consumo_mes_kwh"]
print("\nCorrelación con consumo_mes_kwh:")
print(corr.drop("consumo_mes_kwh").sort_values(ascending=False).round(3))

**Patrón:** `personas` (~0.7) y `superficie_m2` (~0.6) son las únicas con señal estructural real;
el resto aporta ~0. Esto justifica el conjunto de *normalizadores* del consumo esperado.

## 4. Análisis de *data leakage* (paso crítico)

Auditamos el dataset por fugas que darían un modelo con precisión artificialmente perfecta.

In [ ]:
seg = (df.consumo_hora_punta_kwh + df.consumo_hora_valle_kwh + df.consumo_hora_normal_kwh)
print("Fuga 1 | max |punta+valle+normal - consumo|:", round((seg - df.consumo_mes_kwh).abs().max(), 6))
gasto_calc = df.consumo_mes_kwh * df.valor_kwh_clp
print("Fuga 2 | max |gasto - consumo*valor|:", int((df.gasto_mensual_clp - gasto_calc).abs().max()))
print("\nConsumo/persona medio segun 'eficiencia_energetica' del dataset:")
print((df.consumo_mes_kwh / df.personas).groupby(df.eficiencia_energetica).mean().round(1))

**Decisiones:**

- **Fugas 1 y 2**: variables horarias segmentadas y financieras son transformaciones deterministas
  del target → se **excluyen** de las features.
- **Fuga 3**: el consumo/persona es casi idéntico entre Alta/Media/Baja → esa etiqueta **no**
  representa eficiencia real → se **descarta como target**; usamos la definición por residual.

## 5. Transformación de variables y separación normalizadores / palancas

Mapeamos al **contrato de la API (8 campos)** y separamos las variables:

| Grupo | Variables | Rol |
|---|---|---|
| **Normalizadores** (estructura) | `personas`, `superficie_m2`, `cantidad_equipos`, `tipo_inmueble` | Predicen el **consumo esperado** |
| **Palancas** (hábitos) | `uso_horario_pico`, `horas_alto_consumo`, `panel_solar` | Explican el residual → **recomendaciones** |

`uso_horario_pico` y `horas_alto_consumo` se **derivan** del reparto horario real (el reto permite
variables derivadas/simuladas justificadas).

In [ ]:
df["consumo_kwh"]       = df.consumo_mes_kwh.astype(float)
df["cantidad_equipos"]  = df.equipos_electricos.astype(int)
df["tipo_inmueble_num"] = (df.tipo_vivienda == "Departamento").astype(int)   # Departamento=1, Casa=0
df["panel_solar_bool"]  = (~df.panel_solar.str.startswith("N")).astype(int)  # robusto a codificación
share_punta = df.consumo_hora_punta_kwh / df.consumo_mes_kwh
df["uso_horario_pico"]   = (share_punta > share_punta.median()).astype(int)
df["horas_alto_consumo"] = np.clip((share_punta * 24).round(), 1, 12).astype(int)
NORMALIZADORES = ["personas","superficie_m2","cantidad_equipos","tipo_inmueble_num"]
PALANCAS       = ["uso_horario_pico","horas_alto_consumo","panel_solar_bool"]
print("panel_solar:", df.panel_solar_bool.value_counts().to_dict())
print("uso_horario_pico:", df.uso_horario_pico.value_counts().to_dict())
print("horas_alto_consumo (min/med/max):", df.horas_alto_consumo.min(),
      int(df.horas_alto_consumo.median()), df.horas_alto_consumo.max())

## 6. Target: consumo esperado (Modelo A) y residual

**Modelo A** (`LinearRegression`) predice el consumo esperado desde los normalizadores. El
**residual** = real − esperado mide la (in)eficiencia. Cortes por percentil (≈40 y ≈75) →
tres categorías. (Reproduce los umbrales de la API Java del equipo.)

In [ ]:
modelo_a = LinearRegression()
modelo_a.fit(df[NORMALIZADORES], df["consumo_kwh"])
df["consumo_esperado"] = modelo_a.predict(df[NORMALIZADORES])
df["residual"] = df["consumo_kwh"] - df["consumo_esperado"]
print("Modelo A  R^2:", round(modelo_a.score(df[NORMALIZADORES], df["consumo_kwh"]), 3))
print("Coeficientes:", dict(zip(NORMALIZADORES, modelo_a.coef_.round(3))),
      "| intercepto:", round(modelo_a.intercept_, 3))
Q_EFICIENTE, Q_MODERADO = np.percentile(df["residual"], [40, 75])
print("Umbrales residual -> Eficiente <= %.3f < Moderado <= %.3f < Ineficiente" % (Q_EFICIENTE, Q_MODERADO))
def etiquetar(res):
    if res <= Q_EFICIENTE: return "Eficiente"
    if res <= Q_MODERADO:  return "Moderado"
    return "Ineficiente"
df["categoria"] = df["residual"].apply(etiquetar)
print("\nDistribución:", df.categoria.value_counts().to_dict())
print("\nValidación (kWh/persona por categoría, debe crecer):")
print((df.consumo_kwh / df.personas).groupby(df.categoria).mean().round(1))

## 7. Modelos supervisados: Random Forest vs Regresión Logística

Clasificador sobre los **8 campos del contrato**. Operacionaliza la definición por residual en un
modelo servible que además entrega **probabilidad** (`predict_proba`) — el campo `probabilidad` del
contrato, interpretable como confianza del modelo. Elegimos el mejor por F1-macro.

In [ ]:
FEATURES = ["consumo_kwh","personas","superficie_m2","cantidad_equipos",
            "tipo_inmueble_num","uso_horario_pico","horas_alto_consumo","panel_solar_bool"]
X = df[FEATURES].astype(float).values
y = df["categoria"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
modelos = {
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=8,
                                            random_state=RANDOM_STATE, n_jobs=-1),
    "RegresionLogistica": Pipeline([("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, multi_class="multinomial"))]),
}
resultados = {}
for nombre, m in modelos.items():
    m.fit(X_tr, y_tr)
    pred = m.predict(X_te)
    acc = accuracy_score(y_te, pred); f1m = f1_score(y_te, pred, average="macro")
    cv  = cross_val_score(m, X, y, cv=5, scoring="f1_macro").mean()
    resultados[nombre] = {"acc": acc, "f1_macro": f1m, "cv_f1": cv, "modelo": m}
    print(f"=== {nombre} ===  accuracy={acc:.3f}  f1_macro={f1m:.3f}  cv_f1={cv:.3f}")
    print(classification_report(y_te, pred))

In [ ]:
mejor_f1 = max(resultados, key=lambda k: resultados[k]["f1_macro"])
print("Mejor por F1-macro:", mejor_f1)

# --- Modelo SERVIDO: Regresión Logística ---
# Random Forest suele ganar en F1, pero memoriza la etiqueta (determinista respecto al
# residual) y su predict_proba se satura en 0/1 -> la 'probabilidad' saldría siempre 1.0.
# Servimos la Regresión Logística: da probabilidades SUAVES y calibradas (la confianza que
# pide el contrato), sacrificando una diferencia marginal de F1. Si la probabilidad aún
# saliera muy saturada, baja C (p. ej. C=0.3) en el LogisticRegression de la celda anterior.
SERVIDO = "RegresionLogistica"
modelo = resultados[SERVIDO]["modelo"]
CLASES = list(modelo.classes_)   # alfabético: ['Eficiente','Ineficiente','Moderado']
print("Modelo servido:", SERVIDO, "| F1-macro:", round(resultados[SERVIDO]["f1_macro"], 3))
print("Orden de clases (índices del vector de probabilidad):", CLASES)

# Comprobación: la probabilidad máxima por muestra ya NO debe ser siempre 1.0
_pmax = modelo.predict_proba(X_te).max(axis=1)
print("Probabilidad máxima -> min %.2f | media %.2f | max %.2f"
      % (_pmax.min(), _pmax.mean(), _pmax.max()))
cm = confusion_matrix(y_te, modelo.predict(X_te), labels=CLASES)
fig, ax = plt.subplots(figsize=(5, 4)); ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3)); ax.set_xticklabels(CLASES, rotation=45, ha="right")
ax.set_yticks(range(3)); ax.set_yticklabels(CLASES)
ax.set_xlabel("Predicho"); ax.set_ylabel("Real"); ax.set_title(f"Matriz de confusión — {mejor}")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.tight_layout(); plt.show()

## 8. Motor de recomendaciones (reglas sobre las palancas)

Reglas explicables a partir de palancas y categoría: trazables y coherentes con la clasificación.

In [ ]:
def generar_recomendaciones(p, categoria):
    recs = []
    if p["uso_horario_pico"]:
        recs.append("Reducir el uso de equipos durante los horarios pico")
    if p["horas_alto_consumo"] >= 9:
        recs.append("Distribuir las actividades de mayor consumo a lo largo del día")
    if p["cantidad_equipos"] >= 10:
        recs.append("Evaluar equipos con alto consumo energético")
    if not p["panel_solar"] and categoria != "Eficiente":
        recs.append("Evaluar la instalación de paneles solares para autoconsumo")
    if categoria == "Eficiente" and not recs:
        recs.append("Mantener los hábitos actuales de consumo eficiente")
    if not recs:
        recs.append("Monitorear el consumo con un medidor inteligente")
    return recs

## 9. Inferencia end-to-end (contrato JSON) + estimación financiera

Une clasificación + probabilidad + recomendaciones + costo (`consumo_kwh × 0,75`).

In [ ]:
def analizar_consumo(payload: dict) -> dict:
    feat = np.array([[
        payload["consumo_kwh"], payload["personas"], payload["superficie_m2"],
        payload["cantidad_equipos"],
        1 if payload["tipo_inmueble"] == "Departamento" else 0,
        1 if payload["uso_horario_pico"] else 0,
        payload["horas_alto_consumo"],
        1 if payload["panel_solar"] else 0,
    ]], dtype=float)
    proba = modelo.predict_proba(feat)[0]
    idx = int(np.argmax(proba))
    return {
        "categoria": CLASES[idx],
        "probabilidad": round(float(proba[idx]), 2),
        "recomendaciones": generar_recomendaciones(payload, CLASES[idx]),
        "costo_estimado_mensual": round(payload["consumo_kwh"] * TARIFA_REFERENCIA, 2),
    }

## 10. Ejemplos de uso (mínimo 3 exigidos por el reto)

In [ ]:
import json
ejemplos = [
    {"consumo_kwh": 420, "personas": 3, "superficie_m2": 80, "cantidad_equipos": 12,
     "tipo_inmueble": "Casa", "uso_horario_pico": True, "horas_alto_consumo": 10, "panel_solar": False},
    {"consumo_kwh": 180, "personas": 4, "superficie_m2": 95, "cantidad_equipos": 6,
     "tipo_inmueble": "Departamento", "uso_horario_pico": False, "horas_alto_consumo": 6, "panel_solar": True},
    {"consumo_kwh": 300, "personas": 3, "superficie_m2": 70, "cantidad_equipos": 9,
     "tipo_inmueble": "Casa", "uso_horario_pico": True, "horas_alto_consumo": 8, "panel_solar": False},
]
for i, e in enumerate(ejemplos, 1):
    print(f"--- Ejemplo {i} ---")
    print("Entrada:", json.dumps(e, ensure_ascii=False))
    print("Salida :", json.dumps(analizar_consumo(e), ensure_ascii=False, indent=2)); print()

## 11. Prueba de coherencia (categoría ↔ recomendaciones)

Barrido sobre miles de combinaciones: ninguna respuesta sin recomendaciones y ninguna
vivienda *Eficiente* con recomendación contradictoria.

In [ ]:
from itertools import product
incoherencias = 0; total = 0
for consumo, pers, sup, eq, tipo, pico, horas, panel in product(
        [120, 250, 400, 550], [1, 3, 5], [40, 90, 160], [4, 9, 13],
        ["Casa","Departamento"], [False, True], [4, 8, 11], [False, True]):
    total += 1
    r = analizar_consumo({"consumo_kwh":consumo,"personas":pers,"superficie_m2":sup,
        "cantidad_equipos":eq,"tipo_inmueble":tipo,"uso_horario_pico":pico,
        "horas_alto_consumo":horas,"panel_solar":panel})
    if len(r["recomendaciones"]) == 0: incoherencias += 1
    if r["categoria"] == "Eficiente" and \
       "Evaluar la instalación de paneles solares para autoconsumo" in r["recomendaciones"]:
        incoherencias += 1
print(f"Combinaciones probadas: {total}")
print(f"Incoherencias: {incoherencias}  ({100*incoherencias/total:.2f}%)")
assert incoherencias == 0, "Hay respuestas incoherentes"
print("OK: todas las respuestas son coherentes.")

## 12. Serialización a ONNX (consumible por el back-end Java)

Exportamos con `skl2onnx` **sin ZipMap** para que las probabilidades sean un tensor `[N, 3]`.
Servimos la **Regresión Logística** (probabilidades calibradas); su `StandardScaler` queda **dentro** del grafo, así el back envía las features en crudo.

In [ ]:
from skl2onnx import to_onnx
onnx_model = to_onnx(modelo, X_tr[:1].astype(np.float32),
                     options={"zipmap": False}, target_opset=12)
with open("model_clasificador.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())
print("Modelo ONNX guardado: model_clasificador.onnx")

In [ ]:
import onnxruntime as rt
sess = rt.InferenceSession("model_clasificador.onnx", providers=["CPUExecutionProvider"])
in_name = sess.get_inputs()[0].name
print("Input:", in_name, "| Outputs:", [o.name for o in sess.get_outputs()])
muestra = X_te[:5].astype(np.float32)
label_onnx, proba_onnx = sess.run(None, {in_name: muestra})
print("\nLabels ONNX   :", list(label_onnx))
print("Labels sklearn:", list(modelo.predict(muestra)))
print("Máx. diferencia de probabilidad ONNX vs sklearn:",
      float(np.abs(np.array(proba_onnx) - modelo.predict_proba(muestra)).max()))

### Integración con el back-end Java

- **Entrada**: tensor `float32` `[1, 8]` en este orden exacto:
  `[consumo_kwh, personas, superficie_m2, cantidad_equipos, tipo_inmueble_num, uso_horario_pico, horas_alto_consumo, panel_solar]`
  (`tipo_inmueble_num`: Departamento=1, Casa=0; booleanos 1/0).
- **Salidas**: `label` (int64) y `probabilities` (float `[1, 3]`).
- **Orden de clases** (alfabético): `['Eficiente','Ineficiente','Moderado']`.
- En `OnnxModelService.java`: leer `output[0]` → categoría (índice→nombre) y `max(output[1])` →
  `probabilidad`; añadir `probabilidad` y `recomendaciones` al `PredictionResponse` y portar
  `generar_recomendaciones` como método de servicio.

## 13. Cumplimiento de los requisitos del reto

| Requisito | Dónde |
|---|---|
| EDA (exploración y limpieza) | Secciones 2–3 |
| Análisis de patrones | Sección 3 |
| Transformación de variables | Secciones 4–5 |
| Modelos supervisados | Sección 7 (RF vs LogReg) |
| Evaluación con métricas | Sección 7 (accuracy, F1-macro, CV, confusión) |
| Clasificación funcional | Secciones 6–7 |
| **Probabilidad** | Sección 9 (`predict_proba`) |
| Recomendaciones | Sección 8 |
| Estimación financiera (\$0,75/kWh) | Sección 9 |
| Salida JSON | Secciones 9–10 |
| ≥3 ejemplos de uso | Sección 10 |
| Serialización (ONNX) | Sección 12 |
| Manejo de leakage / criterios justificados | Secciones 4, 6 |

**Fuera de este notebook (otros workstreams):** endpoint REST `POST /analisis-energetico`,
validación y manejo de errores en la API, y uso de **OCI** (p. ej. Object Storage para alojar
`model_clasificador.onnx`, u OCI Compute para desplegar la API).